# VJP-delta

`VJPDelta` fits an additive direction at each chosen source layer. It reads a contrast at a later target layer, then uses a vector-Jacobian product (VJP) to map that contrast back to each source layer. The fitted vectors use the usual state-control additive intervention during generation.

VJP-delta follows [Clark, Michael J. (2026), _vjp-steering: contrastive steering vectors from vector-Jacobian products_](https://github.com/wassname/vjp-steering), adapting the [Jacobian lens](https://transformer-circuits.pub/2026/workspace/).

## Setup

This CPU demonstration loads a small Hugging Face Llama checkpoint through the public pipeline API.

In [1]:
from pathlib import Path
import tempfile

from steerability.algorithms.core.steering_pipeline import SteeringPipeline
from steerability.algorithms.state_control.vjp_delta import VJPDelta
from steerability.spipe import SPipe
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "hf-internal-testing/tiny-random-LlamaForCausalLM"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)
fit_data = {
    "positives": ["the cat sat", "the dog ran"],
    "negatives": ["dog ran fast"],
}
control = VJPDelta(
    data=fit_data,
    target_layer=1,
    source_layer_ids=[0],
    skip_first=0,
    strength=0.5,
)
pipeline = SteeringPipeline(
    model=model,
    tokenizer=tokenizer,
    controls=[control],
    model_name_or_path=model_name,
)

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got -1. This may result in unexpected behavior.


[transformers] The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


## Extract and steer

The raw prompts have unequal positive and negative pool sizes. `steer()` extracts the vectors, then binds the standard additive intervention. The stored direction rows are unit norm.

In [2]:
pipeline.steer()
vector = control.export_state()["intervention_0/transform"]
assert set(vector.directions) == {0}
assert all(abs(direction.norm().item() - 1.0) < 1e-5 for direction in vector.directions.values())

reply = pipeline.generate(text="the cat", max_new_tokens=3, do_sample=False)
print({"layers": sorted(vector.directions), "reply": repr(reply)})

{'layers': [0], 'reply': "'agreedָָ'"}


## Freeze and reload

The frozen form contains the fitted vectors as an `ActivationAdapter`. Reloading it resolves the stored additive artifact and does not run another VJP fit.

In [3]:
bundle = Path(tempfile.mkdtemp()) / "vjp_delta_demo"
saved = pipeline.to_spipe().save(bundle)
reloaded = SPipe.load(saved).pipeline()
assert type(reloaded.state_controls[0]).__name__ == "ActivationAdapter"
reloaded.model, reloaded.tokenizer = model, tokenizer
reloaded.steer()
reloaded_reply = reloaded.generate(text="the cat", max_new_tokens=3, do_sample=False)
assert reloaded_reply == reply
print({"bundle": str(saved), "reply_matches": reloaded_reply == reply})

{'bundle': '/tmp/tmpx73lemsj/vjp_delta_demo', 'reply_matches': True}
